<h1>🩺 Biofilter — Report: <code>annotation_master_disease</code></h1>

Everything the bundle knows about a list of diseases: MONDO record, groups, cross-references grouped by the source that issued them, and how many genes ClinGen links to the disease.

### 1. Open a bundle

In [1]:
from biofilter import Biofilter

# A bundle is a directory — the one holding manifest.json.
# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "annotation_master_disease"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)
bf

[INFO] ════════════════════════════════════
[INFO] 🚀 Initializing Biofilter
[INFO]    • Version: 4.3.0
[INFO]    • Debug mode: False
[INFO]    • Config: /Users/andrerico/Works/Sys/biofilter_430/.biofilter.toml
[INFO]    • DB URI: parquet:///Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914
[INFO] ════════════════════════════════════
[INFO] 🔌 Database connection established
[INFO]    • Engine: duckdb+parquet
[INFO]    • Host:   parquet bundle
[INFO]    • DB:     /Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914/tables
[INFO]    • Views:  37 (read-only)
[INFO]    • Time:   161.8 ms
[INFO] ════════════════════════════════════


<Biofilter(db_uri=parquet:///Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914)>

### 2. What the report offers

In [2]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

columns:
  input_value
  input_matched_alias
  entity_id
  disease_id
  disease_label
  disease_description
  omic_status
  disease_groups
  disease_source_system
  disease_data_source
  disease_etl_package_id
  xref_ids_by_source
  clingen_gene_count
  clingen_relationship_count
  entity_relationships_by_group
  total_entity_relationships
  other_aliases
  status
  note

example input:
{'input_data': ['MONDO:0007254', 'breast cancer'], 'include_relationships': True, 'include_xref_summary': True, 'include_clingen_summary': True, 'emit_not_found_rows': True}


In [3]:
print(bf.report.explain(REPORT))

# annotation_master_disease

What the bundle knows about a list of diseases, one row per input.

```bash
biofilter report run --report-name annotation_master_disease \
    --input MONDO:0007254 --input "breast cancer" \
    --output diseases.csv
```

## Input

MONDO ids, labels, synonyms, or cross-reference codes from any source the
bundle carries (`DOID:`, `EFO:`, `ICD10CM:`, …). Matching is
case-insensitive. `--input __ALL__` annotates every disease entity.

## Parameters

| parameter | default | meaning |
| --- | --- | --- |
| `input_data` | required | diseases, or `__ALL__` |
| `include_relationships` | `true` | count relationships by related entity group |
| `include_xref_summary` | `true` | group cross-reference codes by source |
| `include_clingen_summary` | `true` | count ClinGen's gene assertions |
| `emit_not_found_rows` | `true` | keep inputs that resolved to nothing |

## Columns

| column | meaning |
| --- | --- |
| `input_value`, `input_matched_alias` | the input, and the

### 3. Run it

Diseases resolve by MONDO id, by label, or by a cross-reference code from
any source the bundle carries.

In [4]:
diseases = [
    "MONDO:0007254",     # breast cancer, by id
    "Leigh syndrome",    # by label
    "NOT_A_DISEASE",     # kept, with status='not_found'
]

result = bf.report.run(REPORT, input_data=diseases)
df = result.to_pandas()
df[["input_value", "disease_id", "disease_label", "omic_status", "status"]]

[INFO] Report 'annotation_master_disease' produced 3 rows in 0.15s from bundle f47a1c47f3ac95f3.


,input_value,disease_id,disease_label,omic_status,status
0,Leigh syndrome,MONDO:0009723,Leigh syndrome,active,ok
1,MONDO:0007254,MONDO:0007254,breast cancer,active,ok
2,NOT_A_DISEASE,None,None,None,not_found


### 4. The two ClinGen numbers

`clingen_gene_count` counts **distinct genes**; `clingen_relationship_count`
counts assertions. One gene supported by three lines of evidence is one
gene and three assertions — when they diverge, that is why.

In [5]:
df[[
    "input_value",
    "clingen_gene_count",
    "clingen_relationship_count",
    "total_entity_relationships",
]]

,input_value,clingen_gene_count,clingen_relationship_count,total_entity_relationships
0,Leigh syndrome,113,113,119
1,MONDO:0007254,0,0,8
2,NOT_A_DISEASE,0,0,0


ClinGen's share is not the total. A disease can have thousands of
relationships — MONDO's own hierarchy, Reactome, BioGRID — and no ClinGen
genes at all. That means nobody has curated a gene–disease assertion for
it, not that it has no genetic basis.

### 5. Cross-references, grouped by who issued them

In [6]:
for _, row in df[df["status"] == "ok"].iterrows():
    print(row["disease_label"])
    for entry in row["xref_ids_by_source"]:
        print(f"  {entry['source']:<12} {list(entry['ids'])[:4]}")
    print("  groups:", list(row["disease_groups"]))
    print()

Leigh syndrome
  DOID         ['3652']
  GARD         ['0006877']
  ICD10CM      ['G31.82']
  ICD9         ['330.8']
  MEDGEN       ['419518']
  MESH         ['D007888']
  MONDO        ['MONDO:0009723']
  MedDRA       ['10062950']
  NANDO        ['1200175', '2200527']
  NCIT         ['C84814']
  NORD         ['1355']
  OMIM         ['256000']
  Orphanet     ['506']
  SCTID        ['29570005']
  UMLS         ['C2931891']
  icd11.foundation ['672871576']
  groups: ['clingen', 'disease_grouping', 'doid_rare', 'gard_rare', 'ncit_rare', 'nord_rare', 'ordo_disorder', 'orphanet_rare', 'otar', 'rare']

breast cancer
  DOID         ['1612']
  ICD10CM      ['C50']
  ICD9         ['174.8']
  MEDGEN       ['651']
  MONDO        ['MONDO:0007254']
  NCIT         ['C9335']
  SCTID        ['254837009']
  UMLS         ['C0006142']
  icd11.foundation ['1047754165']
  groups: ['otar']



### 6. Every disease in the bundle

In [7]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__", include_clingen_summary=False)
catalog = everything.to_pandas()

print(f"{everything.num_rows:,} diseases in {time.perf_counter() - started:.1f}s")
catalog["status"].value_counts()

[INFO] Report 'annotation_master_disease' produced 36,090 rows in 0.36s from bundle f47a1c47f3ac95f3.


36,090 diseases in 0.5s


status
ok    36090
Name: count, dtype: int64

### 7. Export

CSV by default, with a `.provenance.json` beside it naming the bundle the ids came from.

In [8]:
for path in result.write("annotation_master_disease.csv"):
    print(path)

annotation_master_disease.csv
annotation_master_disease.csv.provenance.json


### 8. The same thing on the command line

```bash
biofilter report run --report-name annotation_master_disease \\
    --input ... \\
    --output out.csv
```

### 9. Quick QA

In [9]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("unresolved inputs:", int((df["status"] == "not_found").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))

missing columns: none
unresolved inputs: 1
bundle: f47a1c47f3ac95f3


,dtype
input_value,object
input_matched_alias,object
entity_id,float64
disease_id,object
disease_label,object
disease_description,object
omic_status,object
disease_groups,object
disease_source_system,object
disease_data_source,object
